In [14]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 

In [3]:
df = pd.read_csv("../data/historical_data.csv", 
                 parse_dates=["created_at", "actual_delivery_time"])

In [5]:
df.shape

(197428, 16)

In [6]:
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


In [7]:
df.dtypes

market_id                                              float64
created_at                                      datetime64[us]
actual_delivery_time                            datetime64[us]
store_id                                                 int64
store_primary_category                                     str
order_protocol                                         float64
total_items                                              int64
subtotal                                                 int64
num_distinct_items                                       int64
min_item_price                                           int64
max_item_price                                           int64
total_onshift_dashers                                  float64
total_busy_dashers                                     float64
total_outstanding_orders                               float64
estimated_order_place_duration                           int64
estimated_store_to_consumer_driving_duration           

In [16]:
df["duration"] = (df["actual_delivery_time"] - df["created_at"]).dt.total_seconds()
print(df["duration"].head(2))

0    3779.0
1    4024.0
Name: duration, dtype: float64


store_primary_category is NaN in rows 2–4, yet row 1 shows store 5477 as mexican. So a store’s missing category can often be recovered from its other rows.

created_at is UTC, and the stores are in US/Pacific. We’ll convert before extracting hour or weekday

In [18]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage":df.isna().sum().div(len(df)).mul(100)
})


missing = missing.sort_values(
    "missing_percentage", ascending=False
)

display(missing)

,missing_count,missing_percentage
total_onshift_dashers,16262,8.236927
total_busy_dashers,16262,8.236927
total_outstanding_orders,16262,8.236927
store_primary_category,4760,2.411006
order_protocol,995,0.503981
market_id,987,0.499929
estimated_store_to_consumer_driving_duration,526,0.266426
actual_delivery_time,7,0.003546
duration,7,0.003546
num_distinct_items,0,0.000000


In [19]:
display(df["duration"].describe())

percentiles = df["duration"].quantile([0.90, 0.99, 0.999])

print("\nTarget percentiles:")
print(percentiles)

print("\nMaximum duration:", df["duration"].max())

count    1.974210e+05
mean     2.908257e+03
std      1.922961e+04
min      1.010000e+02
25%      2.104000e+03
50%      2.660000e+03
75%      3.381000e+03
max      8.516859e+06
Name: duration, dtype: float64


Target percentiles:
0.900    4235.00
0.990    6474.80
0.999    9934.86
Name: duration, dtype: float64

Maximum duration: 8516859.0
